# 08 — Catálogo de algoritmos y evidencia comparativa

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

| Grupo benchmark | Instancia | Diferenciación esperada |
|-----------------|-----------|-------------------------|
| 3D-BPP | `showcase_3d_bpp_instance.json` | best_fit ~27 emp. vs extreme_points ~22 |
| Híbrido (misma instancia 3D) | `showcase_3d_bpp_instance.json` | compaction/LS mueven 8–11 piezas |
| Mejora 3D (misma instancia 3D) | `showcase_3d_bpp_instance.json` | relocation/swap/orientation/bin_reduction refinan layout |
| Container Loading | `showcase_container_loading_instance.json` | weight_aware ~27 vs single_container ~17 |
| Single Container | `showcase_single_container_instance.json` | best_fit ~13 (94% util) vs first_fit ~16 |
| Cartonization | `showcase_cartonization_instance.json` | single-box vs multi_box |
| Palletization | `showcase_palletization_instance.json` | layer_based vs stack_based |
| Stacking-aware | `showcase_stacking_aware_instance.json` | stacking_aware vs stack_based |

Catálogo de **29 algoritmos implementados**, cada uno con endpoint propio ``POST /api/v1/algorithms/{name}/execute`` y **9 grupos de benchmark**. Adaptador ``py3dbp`` activo si la dependencia está instalada.


In [ ]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


In [ ]:
from packing_services.algorithms.registry import get_default_registry
from packing_services.domain.enums import AlgorithmStatus
from _shared.loaders import BENCHMARK_GROUPS
import pandas as pd
from IPython.display import display as ipy_display

registry = get_default_registry()
from packing_services.services.algorithm_input_service import AlgorithmInputService
input_svc = AlgorithmInputService(registry)
impl = registry.list_metadata(status=AlgorithmStatus.IMPLEMENTED)
rows = [{'Algoritmo': m.name, 'Familia': m.algorithm_family.value,
         'Problemas': ', '.join(p.value for p in m.problem_types),
         'Input': input_svc.input_schema_for(m),
         'Endpoint': input_svc.execution_endpoint(m.name)} for m in impl]
ipy_display(pd.DataFrame(rows).style.hide(axis='index'))


In [ ]:
print('═══ Demo: ejecutar un algoritmo vía servicio propio ═══')
from _shared.loaders import build_algorithm_execute_input
demo_algo = 'extreme_points_3d'
payload = build_algorithm_execute_input(demo_algo, problem_type='3D_BPP')
print(f'POST /api/v1/algorithms/{demo_algo}/execute')
display.show_execute_result(runners.run_algorithm_execute(demo_algo, payload))


In [ ]:
print('═══ Evidencia comparativa: 9 benchmarks homogéneos ═══')
for group in BENCHMARK_GROUPS:
    display.run_and_show_showcase_benchmark(group)


In [ ]:
by = {}
for m in registry.list_metadata(): by[m.status.value] = by.get(m.status.value, 0) + 1
fig = viz.plot_algorithm_catalog_counts(by.get('implemented', 0), by.get('adapter', 0), by.get('future', 0))
plt.show()
fig = viz.plot_roadmap_phases(); plt.show()


**Siguiente:** `09_preparacion_espacio_de_datos.ipynb`
